# C04 · Post-LASSO Instrumental Variables

**Paper:** Belloni, A., Chen, D., Chernozhukov, V., & Hansen, C. (2012). Sparse models and methods for optimal instruments. *Econometrica, 80*(6), 2369–2429.

**Key idea:** When the instrument set Z is high-dimensional, LASSO selects relevant instruments in the first stage. Post-LASSO OLS (unpenalised re-fit on selected instruments) corrects for LASSO shrinkage bias, yielding near-optimal 2SLS.

In [ ]:
import numpy as np
import pandas as pd
from empirlab.causal import PostLassoIV
np.random.seed(42)

## 1 · Simulate High-Dimensional IV DGP

- n=500, q=30 candidate instruments, only 3 are relevant
- D is endogenous: correlated with the structural error

In [ ]:
n, p, q = 500, 10, 30
rng = np.random.default_rng(0)
Z = rng.standard_normal((n, q))
X = rng.standard_normal((n, p))
pi_true = np.zeros(q); pi_true[:3] = [1.5, 1.0, 0.8]
nu = rng.standard_normal(n)
D = Z @ pi_true + X[:, 0] + nu * 0.5
beta_true = 2.0
eps = 0.5 * nu + rng.standard_normal(n)
Y = beta_true * D + X[:, 0] + eps
print(f"True beta = {beta_true}, n={n}, q={q}, relevant instruments = 3")

## 2 · OLS (biased baseline)

In [ ]:
W_ols = np.hstack([D[:, None], X, np.ones((n, 1))])
coef_ols = np.linalg.lstsq(W_ols, Y, rcond=None)[0]
print(f"OLS coefficient: {coef_ols[0]:.4f}  (truth = {beta_true})")
print(f"OLS bias: {coef_ols[0] - beta_true:+.4f}")

## 3 · Post-LASSO 2SLS

In [ ]:
iv = PostLassoIV(penalty='CV')
iv.fit(X, Y, D, Z)
print(iv.summary().to_string())
print(f"\nSelected instrument indices: {iv.selected_instruments_}")
print(f"First-stage F-stat: {iv.f_stat_:.1f}  (rule-of-thumb: >10)")

## 4 · Results Comparison

In [ ]:
results = pd.DataFrame({
    'Method':      ['OLS (biased)', 'Post-LASSO IV', 'True'],
    'Coefficient': [coef_ols[0], iv.coef_, beta_true],
    'Bias':        [coef_ols[0]-beta_true, iv.coef_-beta_true, 0.0],
})
print(results.to_string(index=False))